<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 18


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс CreditCard в C#, который будет представлять информацию
о кредитных картах. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма

Требования к базовому классу CreditCard:
• Атрибуты: Номер карты (CardNumber), Холдера (HolderName), Срок
действия (ExpiryDate).
• Методы:
o GetInfo(): метод для получения информации о кредитной карте в виде
строки.
o Pay(): метод для оплаты покупки с использованием карты.
o CheckBalance(): метод для проверки баланса на карте.
Требования к производным классам:
1. GoldCard (GoldCreditCard): Должен содержать дополнительные атрибуты,
такие как Бесплатные бонусные мили (BonusMiles). Метод Pay() должен быть
переопределен для добавления информации о получении бонусных миль
при оплате покупки.
2. PremiumCard (PremiumCreditCard): Должен содержать дополнительные
атрибуты, такие как Ассистент поддержки (SupportAssistant).
Метод CheckBalance() должен быть переопределен для предоставления
возможности обратиться за помощью к ассистенту поддержки в случае
проблем с балансом.
3. CorporateCard (CorporateCreditCard) (если требуется третий класс): Должен
содержать дополнительные атрибуты, такие как Компания (Company).
Метод GetInfo() должен быть переопределен для включения информации о
компании в описании карты.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
public interface IReportable
{
    void GenerateReport();
}

public interface IBonusProgram
{
    decimal BonusBalance { get; set; }
    void AddBonus(decimal amount);
    void RedeemBonus(decimal amount);
}

public class CreditCard
{
    public string CardNumber { get; set; }
    public string HolderName { get; set; }
    public DateTime ExpiryDate { get; set; }
    public string Currency { get; set; } = "RUB";
    public string PinCode { get; private set; }
    public bool IsBlocked { get; set; } = false;

    private decimal _balance;
    public decimal Balance
    {
        get => _balance;
        set
        {
            if (value < 0)
            {
                Console.WriteLine("Ошибка: баланс не может быть отрицательным.");
            }
            else
            {
                _balance = value;
            }
        }
    }

    public CreditCard(string cardNumber, string holderName, DateTime expiryDate, string currency = "RUB", string pinCode = "0000", decimal initialBalance = 0)
    {
        CardNumber = cardNumber;
        HolderName = holderName;
        ExpiryDate = expiryDate;
        Currency = currency;
        PinCode = pinCode;
        Balance = initialBalance;
    }

    public virtual string GetInfo()
    {
        return $"Номер карты: {CardNumber}, Холдер: {HolderName}, Срок действия: {ExpiryDate:MM.yyyy}, Баланс: {Balance:C} {Currency}, Статус: {(IsBlocked ? "Заблокирована" : "Активна")}";
    }

    public virtual void Pay(decimal amount)
    {
        if (IsBlocked) 
        {    
            Console.WriteLine($"Ошибка: Карта {CardNumber} заблокирована. Оплата невозможна."); 
            return;
        }

        if (Balance >= amount)
        {
            Balance -= amount;
            Console.WriteLine($"Оплата покупки на сумму {amount:C} с карты {CardNumber}. Остаток: {Balance:C}");
        }
        else
        {
            Console.WriteLine($"Недостаточно средств на карте {CardNumber} для оплаты {amount:C}");
        }
    }

    public virtual void CheckBalance()
    {
        Console.WriteLine($"Проверка баланса карты {CardNumber}: {Balance:C} {Currency}");
    }

    public bool ChangePin(string oldPin, string newPin)
    {
        if (PinCode == oldPin)
        {
            PinCode = newPin;
            Console.WriteLine($"ПИН-код для карты {CardNumber} успешно изменен.");
            return true;
        }
        Console.WriteLine("Неверный текущий ПИН-код.");
        return false;
    }

    public void BlockCard()
    {
        IsBlocked = true;
        Console.WriteLine($"Карта {CardNumber} заблокирована");
    }

    public void UnBlockCard()
    {
        IsBlocked = false;
        Console.WriteLine($"Карта {CardNumber} разблокирована");
    }

    public bool TransferTo(CreditCard targetCard, decimal amount)
    {
        if (IsBlocked || targetCard == null || targetCard.IsBlocked)
        {
            Console.WriteLine($"[Перевод] Операция невозможна: одна из карт заблокирована или не существует.");
            return false;
        }

        if (Balance >= amount)
        {
            Balance -= amount;
            targetCard.Balance += amount;
            Console.WriteLine($"[Перевод] Успешно переведено {amount:C} с карты {CardNumber} на карту {targetCard.CardNumber}.");
            return true;
        }

        Console.WriteLine($"[Перевод] Недостаточно средств на карте {CardNumber} для перевода {amount:C}.");
        return false;
    }
}

public class GoldenCard : CreditCard, IBonusProgram
{
    public int BonusMiles { get; set; }
    public decimal CashbackRate { get; set; } = 0.05m;
    public decimal BonusBalance { get; set; }

    public GoldenCard(string cardNumber, string holderName, DateTime expiryDate, decimal initialBalance = 0, int bonusMiles = 0)
        : base(cardNumber, holderName, expiryDate, "RUB", "0000", initialBalance)
    {
        BonusMiles = bonusMiles;
    }

    public override void Pay(decimal amount)
    {
        if (Balance >= amount && !IsBlocked)
        {
            base.Pay(amount);
            int earnedMiles = (int)(amount / 10);
            BonusMiles += earnedMiles;
            AddBonus(amount * CashbackRate);
            Console.WriteLine($"За покупку начислено {earnedMiles} бонусных миль. Всего миль: {BonusMiles}");
        }
        else
        {
            Console.WriteLine($"Не удалось совершить покупку по Gold-карте {CardNumber}.");
        }
    }

    public void AddBonus(decimal amount)
    {
        BonusBalance += amount;
        Console.WriteLine($"|Кешбек| Начислено {amount:C} на бонусный счет");
    }

    public void RedeemBonus(decimal amount)
    {
        if (BonusBalance >= amount)
        {
            BonusBalance -= amount;
            Balance += amount;
            Console.WriteLine($"|Кешбэк| Переведено {amount:C} с бонусного счета на основной баланс");
        }
    }
}

public class PremiumCard : CreditCard, IReportable
{
    public string SupportAssistant { get; set; }
    public bool IsInsured { get; set; }
    public decimal CreditLimit { get; set; } = 300000;

    public PremiumCard(string cardNumber, string holderName, DateTime expiryDate, decimal initialBalance = 0, string supportAssistant = "Персональный менеджер", bool isInsured = true)
        : base(cardNumber, holderName, expiryDate, "RUB", "0000", initialBalance)
    {
        SupportAssistant = supportAssistant;
        IsInsured = isInsured;
    }

    public override void CheckBalance()
    {
        base.CheckBalance();
        if (Balance < 1000)
        {
            Console.WriteLine($"[Внимание] Ваш баланс ниже лимита. Ваш ассистент {SupportAssistant} готов помочь с пополнением.");
        }
        else
        {
            Console.WriteLine($"Ваш личный ассистент поддержки — {SupportAssistant}.");
        }
    }

    public void DirectCallSupport()
    {
        Console.WriteLine($"Соединяем с ассистентом {SupportAssistant} по выделенной линии...");
    }

    public void GenerateReport()
    {
        Console.WriteLine($"--- Премиальный отчет по карте {CardNumber} ---");
        Console.WriteLine($"Владелец: {HolderName}");
        Console.WriteLine($"Закрепленный менеджер: {SupportAssistant}");
        Console.WriteLine($"Кредитный лимит: {CreditLimit:C}");
        Console.WriteLine($"Страхование средств: {(IsInsured ? "Активно (покрывает до 1 млн ₽)" : "Не подключено")}");
    }
}

public class CorporateCard : CreditCard
{
    public string Company { get; set; }

    public CorporateCard(string cardNumber, string holderName, DateTime expiryDate, string company, decimal initialBalance = 0)
        : base(cardNumber, holderName, expiryDate, "RUB", "0000", initialBalance)
    {
        Company = company;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $", Компания: {Company}";
    }

    public void TransferBonusTo(GoldenCard targetGoldCard, int miles)
    {
        targetGoldCard.BonusMiles += miles;
        Console.WriteLine($"Компания {Company} перевела {miles} бонусных миль на карту {targetGoldCard.CardNumber}.");
    }
}

GoldenCard goldCard = new GoldenCard("1234-5678-9012-3456", "Марк Серков", new DateTime(2029, 12, 1), 5000, 200);
PremiumCard premiumCard = new PremiumCard("1111-3333-5555-8888", "Андрей Зелинский", new DateTime(2029, 12, 1), 800, "Анна", true);
CorporateCard corporateCard = new CorporateCard("9999-9999-9999-9998", "Дмитрий Ицков", new DateTime(2033, 6, 1), "ООО RRepiv", 20000);

Console.WriteLine(goldCard.GetInfo());
Console.WriteLine(premiumCard.GetInfo());
Console.WriteLine(corporateCard.GetInfo());

Console.WriteLine();
goldCard.Pay(1500);

Console.WriteLine();
goldCard.RedeemBonus(50);

Console.WriteLine();
premiumCard.CheckBalance();

Console.WriteLine();
premiumCard.DirectCallSupport();
premiumCard.GenerateReport();

Console.WriteLine();
corporateCard.TransferTo(goldCard, 3000);
goldCard.TransferTo(premiumCard, 1000);
corporateCard.TransferBonusTo(goldCard, 500);

Console.WriteLine();
corporateCard.ChangePin("0000", "4321");
corporateCard.BlockCard();
corporateCard.Pay(500);
corporateCard.UnBlockCard();

Console.WriteLine();
goldCard.CheckBalance();
premiumCard.CheckBalance();
corporateCard.CheckBalance();

Номер карты: 1234-5678-9012-3456, Холдер: Марк Серков, Срок действия: 12.2029, Баланс: 5 000,00 ₽ RUB, Статус: Активна
Номер карты: 1111-3333-5555-8888, Холдер: Андрей Зелинский, Срок действия: 12.2029, Баланс: 800,00 ₽ RUB, Статус: Активна
Номер карты: 9999-9999-9999-9998, Холдер: Дмитрий Ицков, Срок действия: 06.2033, Баланс: 20 000,00 ₽ RUB, Статус: Активна, Компания: ООО RRepiv

Оплата покупки на сумму 1 500,00 ₽ с карты 1234-5678-9012-3456. Остаток: 3 500,00 ₽
|Кешбек| Начислено 75,00 ₽ на бонусный счет
За покупку начислено 150 бонусных миль. Всего миль: 350

|Кешбэк| Переведено 50,00 ₽ с бонусного счета на основной баланс

Проверка баланса карты 1111-3333-5555-8888: 800,00 ₽ RUB
[Внимание] Ваш баланс ниже лимита. Ваш ассистент Анна готов помочь с пополнением.

Соединяем с ассистентом Анна по выделенной линии...
--- Премиальный отчет по карте 1111-3333-5555-8888 ---
Владелец: Андрей Зелинский
Закрепленный менеджер: Анна
Кредитный лимит: 300 000,00 ₽
Страхование средств: Активно (п